In [4]:
import pandas as pd
import numpy as np

In [26]:
print("--- Carregando todos os arquivos ---")
# Carrega cada arquivo CSV para um DataFrame separado
df_admissao = pd.read_csv('ADMISSÃO ABRIL - Planilha1.csv')
df_afastamentos = pd.read_csv('AFASTAMENTOS - Planilha1.csv')
df_aprendiz = pd.read_csv('APRENDIZ - Planilha1.csv')
df_ativos = pd.read_csv('ATIVOS.xlsx - Planilha1.csv')
# O 'header=1' pula a primeira linha do arquivo de dias úteis, que era um cabeçalho extra
df_dias_uteis = pd.read_csv('Base dias uteis - Planilha1.csv', header=1, usecols=[0, 1])
df_sindicato_valor = pd.read_csv('Base sindicato x valor - Planilha1.csv')
df_desligados = pd.read_csv('DESLIGADOS - Planilha1.csv')
df_estagio = pd.read_csv('ESTÁGIO - Planilha1.csv')
df_exterior = pd.read_csv('EXTERIOR - Planilha1.csv')
df_ferias = pd.read_csv('FÉRIAS - Planilha1.csv')
df_vr_mensal = pd.read_csv('VR MENSAL 05.2025 - Planilha1.csv')

--- Carregando todos os arquivos ---


In [27]:
print("\n--- 2. Limpeza e Padronização dos Dados ---")
# Remove espaços em branco dos nomes das colunas em todos os DataFrames
# A linha abaixo foi corrigida e não tem mais recuo indevido
for df in [df_admissao, df_afastamentos, df_aprendiz, df_ativos, df_desligados, df_estagio, df_exterior, df_ferias]:
    df.columns = df.columns.str.strip()

    # Renomeia as colunas para um padrão consistente
df_ativos.rename(columns={'MATRICULA': 'Matricula', 'TITULO DO CARGO': 'Cargo', 'Sindicato': 'Sindicato'}, inplace=True)
df_admissao.rename(columns={'MATRICULA': 'Matricula', 'Admissão': 'Data Admissao', 'Cargo':'Cargo'}, inplace=True)
df_desligados.rename(columns={'MATRICULA': 'Matricula', 'DATA DEMISSÃO': 'Data Demissao', 'COMUNICADO DE DESLIGAMENTO': 'Comunicado Desligamento'}, inplace=True)
df_ferias.rename(columns={'MATRICULA': 'Matricula', 'DIAS DE FÉRIAS': 'Dias Ferias'}, inplace=True)
df_afastamentos.rename(columns={'MATRICULA': 'Matricula'}, inplace=True)
df_aprendiz.rename(columns={'MATRICULA': 'Matricula'}, inplace=True)
df_estagio.rename(columns={'MATRICULA': 'Matricula'}, inplace=True)
df_exterior.rename(columns={'Cadastro': 'Matricula'}, inplace=True)

    # Limpa os arquivos de Sindicato e Dias Úteis
df_sindicato_valor.columns = ['SINDICATO', 'VALOR']
df_sindicato_valor['SINDICATO'] = df_sindicato_valor['SINDICATO'].str.strip()
df_sindicato_valor.dropna(inplace=True)
df_dias_uteis.columns = ['SINDICATO', 'DIAS_UTEIS']
df_dias_uteis.dropna(inplace=True)


--- 2. Limpeza e Padronização dos Dados ---


In [29]:
print("\n--- 3. Consolidação da Base e Aplicação de Exclusões ---")
base_colaboradores = pd.concat([
    df_ativos[['Matricula', 'Cargo', 'Sindicato']],
    df_admissao[['Matricula', 'Cargo']]
    ], ignore_index=True).drop_duplicates(subset='Matricula')

matriculas_excluir = set(
    df_afastamentos['Matricula'].tolist() +
    df_aprendiz['Matricula'].tolist() +
    df_estagio['Matricula'].tolist() +
    df_exterior['Matricula'].tolist()
    )
base_elegiveis = base_colaboradores[~base_colaboradores['Matricula'].isin(matriculas_excluir)].copy()

print(f"Base consolidada com {base_elegiveis.shape[0]} colaboradores elegíveis.")


--- 3. Consolidação da Base e Aplicação de Exclusões ---
Base consolidada com 1801 colaboradores elegíveis.


In [30]:
print("\n--- 4. Enriquecimento da Base de Dados ---")
    # Converte colunas de texto para data ANTES de qualquer operação
df_admissao['Data Admissao'] = pd.to_datetime(df_admissao['Data Admissao'], errors='coerce')
df_desligados['Data Demissao'] = pd.to_datetime(df_desligados['Data Demissao'], errors='coerce')

    # Junta todas as informações na base principal
base_elegiveis = pd.merge(base_elegiveis, df_desligados, on='Matricula', how='left')
base_elegiveis = pd.merge(base_elegiveis, df_ferias[['Matricula', 'Dias Ferias']], on='Matricula', how='left')
base_elegiveis = pd.merge(base_elegiveis, df_admissao[['Matricula', 'Data Admissao']], on='Matricula', how='left')

def map_sindicato_to_estado(sindicato_str):
  s_str = str(sindicato_str).upper()
  if 'PR' in s_str: return 'Paraná'
  if 'RS' in s_str: return 'Rio Grande do Sul'
  if 'SP' in s_str: return 'São Paulo'
  if 'RJ' in s_str: return 'Rio de Janeiro'
  return None

base_elegiveis['Estado Sindicato'] = base_elegiveis['Sindicato'].apply(map_sindicato_to_estado)
df_sindicato_valor.rename(columns={'SINDICATO':'Estado Sindicato', 'VALOR':'Valor Diario'}, inplace=True)
df_dias_uteis['Estado Sindicato'] = df_dias_uteis['SINDICATO'].apply(map_sindicato_to_estado)

base_elegiveis = pd.merge(base_elegiveis, df_sindicato_valor, on='Estado Sindicato', how='left')
base_elegiveis = pd.merge(base_elegiveis, df_dias_uteis[['Estado Sindicato', 'DIAS_UTEIS']], on='Estado Sindicato', how='left')


--- 4. Enriquecimento da Base de Dados ---


In [31]:
print("\n--- 5. Aplicação das Regras de Negócio (Versão Otimizada) ---")
    # Garante que as colunas numéricas não tenham valores nulos
base_elegiveis['DIAS_UTEIS'] = base_elegiveis['DIAS_UTEIS'].fillna(0)
base_elegiveis['Valor Diario'] = base_elegiveis['Valor Diario'].fillna(0)
base_elegiveis['Dias Ferias'] = base_elegiveis['Dias Ferias'].fillna(0)
base_elegiveis['dias_a_pagar'] = base_elegiveis['DIAS_UTEIS']

    # Regra de Admissão
period_end = pd.to_datetime('2025-08-15')
mask_admitidos = base_elegiveis['Data Admissao'].notna()
if not base_elegiveis.loc[mask_admitidos].empty:
  begindates_adm = base_elegiveis.loc[mask_admitidos, 'Data Admissao'].values.astype('datetime64[D]')
  enddate_adm = np.datetime64(period_end, 'D')
  dias_proporcionais_adm = np.busday_count(begindates_adm, enddate_adm) + 1
  base_elegiveis.loc[mask_admitidos, 'dias_a_pagar'] = dias_proporcionais_adm

    # Regra de Desligamento
period_start = pd.to_datetime('2025-07-16')
mask_desligamento_zera = (base_elegiveis['Data Demissao'].notna()) & (base_elegiveis['Comunicado Desligamento'] == 'OK') & (base_elegiveis['Data Demissao'].dt.day <= 15)
base_elegiveis.loc[mask_desligamento_zera, 'dias_a_pagar'] = 0

mask_desligamento_proporcional = (base_elegiveis['Data Demissao'].notna()) & (~mask_desligamento_zera)
if not base_elegiveis.loc[mask_desligamento_proporcional].empty:
  begindate_des = np.datetime64(period_start, 'D')
  enddates_des = base_elegiveis.loc[mask_desligamento_proporcional, 'Data Demissao'].values.astype('datetime64[D]')
  dias_proporcionais_des = np.busday_count(begindate_des, enddates_des) + 1
  base_elegiveis.loc[mask_desligamento_proporcional, 'dias_a_pagar'] = np.minimum(base_elegiveis.loc[mask_desligamento_proporcional, 'dias_a_pagar'], dias_proporcionais_des)

    # Regra de Férias
base_elegiveis['dias_a_pagar'] = base_elegiveis['dias_a_pagar'] - base_elegiveis['Dias Ferias']
base_elegiveis['dias_a_pagar'] = base_elegiveis['dias_a_pagar'].clip(lower=0)


--- 5. Aplicação das Regras de Negócio (Versão Otimizada) ---


In [35]:
print("\n--- 6. Cálculo dos Valores Finais ---")
# Convert 'Valor Diario' to numeric, handling potential 'R$' and commas
base_elegiveis['Valor Diario'] = base_elegiveis['Valor Diario'].astype(str).str.replace('R$', '', regex=False).str.replace(',', '.', regex=False)
base_elegiveis['Valor Diario'] = pd.to_numeric(base_elegiveis['Valor Diario'], errors='coerce').fillna(0)

base_elegiveis['Valor Total VR'] = base_elegiveis['dias_a_pagar'] * base_elegiveis['Valor Diario']
base_elegiveis['Custo Empresa'] = base_elegiveis['Valor Total VR'] * 0.80
base_elegiveis['Desconto Colaborador'] = base_elegiveis['Valor Total VR'] * 0.20

print("\n--- 7. Geração do Relatório Final ---")
base_elegiveis['NOME DO PROFISSIONAL'] = 'NOME NÃO DISPONÍVEL'
base_elegiveis['CPF'] = '000.000.000-00'

output_df = base_elegiveis[[
    'Matricula', 'NOME DO PROFISSIONAL', 'CPF',
    'Valor Total VR', 'Custo Empresa', 'Desconto Colaborador'
]].copy()

output_df.rename(columns={
    'Matricula': 'MATRICULA', 'Valor Total VR': 'VALOR TOTAL DO BENEFÍCIO',
    'Custo Empresa': 'VALOR PAGO PELA EMPRESA', 'Desconto Colaborador': 'VALOR A SER DESCONTADO DO PROFISSIONAL'
}, inplace=True)

output_filename = 'VR MENSAL 05.2025 vfinal.csv'

try:
  output_df.to_csv(output_filename, index=False, decimal=',', sep=';')

  print(f"\n>>> SUCESSO! O arquivo final '{output_filename}' foi gerado.")
  print("\nAmostra do resultado:")
  print(output_df.head())

except FileNotFoundError as e:
  print(f"\nERRO: Arquivo não encontrado. Verifique se o nome do arquivo está correto: {e.fileName}")
except Exception as e:
  print(f"\nOcorreu um erro inesperado durante a execução: {e}")


--- 6. Cálculo dos Valores Finais ---

--- 7. Geração do Relatório Final ---

>>> SUCESSO! O arquivo final 'VR MENSAL 05.2025 vfinal.csv' foi gerado.

Amostra do resultado:
   MATRICULA NOME DO PROFISSIONAL             CPF  VALOR TOTAL DO BENEFÍCIO  \
0      34941  NOME NÃO DISPONÍVEL  000.000.000-00                     770.0   
1      34941  NOME NÃO DISPONÍVEL  000.000.000-00                     735.0   
2      34941  NOME NÃO DISPONÍVEL  000.000.000-00                     770.0   
3      34941  NOME NÃO DISPONÍVEL  000.000.000-00                     735.0   
4      24401  NOME NÃO DISPONÍVEL  000.000.000-00                     770.0   

   VALOR PAGO PELA EMPRESA  VALOR A SER DESCONTADO DO PROFISSIONAL  
0                    616.0                                   154.0  
1                    588.0                                   147.0  
2                    616.0                                   154.0  
3                    588.0                                   147.0  
4     